In [8]:
import nibabel as nib
import numpy as np

# Load ground truth & reconstruction
gt_img =  nib.load("/projectnb/ec500kb/projects/Fall_2025_Projects/Project_4_VesselFM/data/nnUNet_preprocessed/Dataset001_nnunet/gt_segmentations/case_022.nii.gz")
recon_img = nib.load("/projectnb/ec500kb/projects/Project_4/Graph_Creations/Everything_For_Docker/output2/pred_case_022_0000.nii.gz")

gt = gt_img.get_fdata().astype(int)
recon = recon_img.get_fdata().astype(int)

classes = {1: "artery", 2: "vein"}

# Build confusion matrix (multi-class)
max_class = max(gt.max(), recon.max())
cm = np.zeros((max_class+1, max_class+1), dtype=int)
for c in range(max_class+1):
    for r in range(max_class+1):
        cm[c, r] = np.sum((gt == c) & (recon == r))

# For each class, compute TP, FP, FN, TN, and FPR/FNR
results = {}
total_voxels = gt.size

for cls, name in classes.items():
    TP = cm[cls, cls]
    FP = cm[:, cls].sum() - TP         # predicted cls but GT is not cls
    FN = cm[cls, :].sum() - TP         # GT is cls but predicted not cls
    TN = total_voxels - (TP + FP + FN)

    # Rates
    FPR = FP / (FP + TN + 1e-8)   # how often non-cls voxels wrongly predicted as cls 
    FNR = FN / (TP + FN + 1e-8)   # how often cls voxels missed

    results[name] = {
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "FPR": FPR,
        "FNR": FNR
    }

for name, vals in results.items():
    print(f"\nClass = {name}")
    print(" TP:", vals["TP"])
    print(" FP:", vals["FP"])
    print(" FN:", vals["FN"])
    print(" TN:", vals["TN"])
    print(f" False Positive Rate (FPR): {vals['FPR']:.4f}")
    print(f" False Negative Rate (FNR): {vals['FNR']:.4f}")



Class = artery
 TP: 520046
 FP: 253824
 FN: 198430
 TN: 86059508
 False Positive Rate (FPR): 0.0029
 False Negative Rate (FNR): 0.2762

Class = vein
 TP: 490651
 FP: 284288
 FN: 233590
 TN: 86023279
 False Positive Rate (FPR): 0.0033
 False Negative Rate (FNR): 0.3225
